In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import glob
import matplotlib.ticker as ticker
import numpy as np
import re
from pathlib import Path
from mpl_toolkits.axes_grid1.inset_locator import mark_inset
import json
from matplotlib.patches import Patch
import ast
import matplotlib.patches as mpatches

In [2]:
import re

def parse_taskset_cores(item):
    """
    Extract cores from a Kubernetes pod spec command like:
    taskset -c 0-7 ./run ...
    taskset -c 1 ./run ...
    taskset -c 1-3 ./run ...

    Returns a list of ints, e.g. [0, 1, 2, 3].
    """
    containers = item["spec"].get("containers", [])
    if not containers:
        return []

    args = containers[0].get("args", [])
    command_string = " ".join(args)

    match = re.search(r"taskset\s+-c\s+([0-9,\-]+)", command_string)
    if not match:
        return []

    core_string = match.group(1)

    cores = []
    for part in core_string.split(","):
        if "-" in part:
            start, end = map(int, part.split("-"))
            cores.extend(range(start, end + 1))
        else:
            cores.append(int(part))

    return cores

In [3]:
from pathlib import Path

parent = Path("results/openevolve-best")
all_dfs = []

for subdir in parent.iterdir():
    if subdir.is_dir():
        print(str(subdir))

        with open(str(subdir) + "/pods.json") as f:
            pods = json.load(f)

        rows = []

        for item in pods["items"]:

            name = item["metadata"]["name"]

            # job name
            if "job-name" in item["metadata"].get("labels", {}):
                job = item["metadata"]["labels"]["job-name"]
            else:
                job = name
            if job == "some-memcached":
                continue

            machine = item["spec"]["nodeName"]

            cores = parse_taskset_cores(item)

            status = item["status"]["containerStatuses"][0]["state"]

            if "terminated" in status:

                start = status["terminated"]["startedAt"]
                end = status["terminated"]["finishedAt"]

            elif "running" in status:

                start = status["running"]["startedAt"]
                end = None

            else:
                continue

            rows.append({
                "job": job,
                "machine": machine,
                "cores": cores,
                "ts_start": start,
                "ts_end": end,
            })

        schedule_df = pd.DataFrame(rows)

        schedule_df["ts_start"] = pd.to_datetime(schedule_df["ts_start"])
        schedule_df["ts_end"] = pd.to_datetime(schedule_df["ts_end"])

        schedule_df["ts_start"] = schedule_df["ts_start"].apply(lambda x: x.timestamp())
        schedule_df["ts_end"] = schedule_df["ts_end"].apply(lambda x: x.timestamp())

        schedule_df["machine"] = schedule_df["machine"].str.replace(r"(core).*", r"\1", regex=True)

        schedule_df["width"] = schedule_df["ts_end"] - schedule_df["ts_start"]

        # get total runtime
        total_start = min(schedule_df["ts_start"])
        total_end = max(schedule_df["ts_end"])
        total_width = total_end - total_start

        schedule_df.loc[len(schedule_df)] = {"job": "total", "machine": "x", "cores": "x", "ts_start": total_start, "ts_end": total_end, "width": total_width}

        all_dfs.append(schedule_df)

schedule_df = pd.concat(all_dfs)
schedule_df

results/openevolve-best/2026-05-04_21:31:34
results/openevolve-best/2026-05-12_20:50:19
results/openevolve-best/2026-05-04_21:22:22


,job,machine,cores,ts_start,ts_end,width
0,parsec-barnes,node-a-8core,"[4, 5, 6, 7]",1.777923e+09,1.777923e+09,63.0
1,parsec-blackscholes,node-a-8core,"[0, 1, 2, 3]",1.777923e+09,1.777923e+09,73.0
2,parsec-canneal,node-a-8core,"[0, 1, 2, 3]",1.777923e+09,1.777923e+09,176.0
3,parsec-freqmine,node-a-8core,"[4, 5, 6, 7]",1.777923e+09,1.777923e+09,180.0
4,parsec-radix,node-a-8core,[0],1.777923e+09,1.777923e+09,63.0
5,parsec-streamcluster,node-b-4core,"[1, 2, 3]",1.777923e+09,1.777923e+09,219.0
6,parsec-vips,node-a-8core,"[4, 5, 6, 7]",1.777923e+09,1.777923e+09,40.0
7,total,x,x,1.777923e+09,1.777923e+09,323.0
0,parsec-barnes,node-a-8core,"[4, 5, 6, 7]",1.778612e+09,1.778612e+09,43.0
1,parsec-blackscholes,node-a-8core,"[0, 1, 2, 3]",1.778612e+09,1.778612e+09,39.0


In [4]:
result = (
    schedule_df.groupby("job")["width"]
      .agg(mean_value="mean", std_value="std")
)

result

,mean_value,std_value
job,,
parsec-barnes,57.000000,12.165525
parsec-blackscholes,64.000000,21.931712
parsec-canneal,164.333333,17.672955
parsec-freqmine,158.000000,39.849718
parsec-radix,54.333333,15.011107
parsec-streamcluster,216.333333,6.429101
parsec-vips,33.333333,11.547005
total,290.666667,56.002976
